In [ ]:
# Lecture 9 — Lazy Evaluation · .explain() demo
# Spark 101 · Module 3 · Lecture 9
#
# Import into a Databricks workspace as a notebook.

In [ ]:
# Load a real dataset — many columns, many rows.
df = spark.read.table("samples.nyctaxi.trips")
df.columns

In [ ]:
# A query that selects a couple of columns and filters — the classic pushdown scenario.
from pyspark.sql.functions import col

result = (
    df.filter(col("trip_distance") > 5)
      .select("passenger_count", "fare_amount")
)

In [ ]:
# The physical plan — what actually runs.
# Look for PushedFilters (predicate pushdown) and ReadSchema (column pruning).
result.explain()

In [ ]:
# All four plans — Parsed, Analyzed, Optimized, Physical.
result.explain(True)

In [ ]:
# The "formatted" version — cleaner for reading in the notebook.
result.explain("formatted")

In [ ]:
# Column pruning: selecting two columns reads far less than selecting all.
from pyspark.sql.functions import col
df.select("passenger_count", "fare_amount").filter(col("fare_amount") > 10).explain()

In [ ]:
# Catalyst reorders: a filter written AFTER a select still pushes to the scan.
df.select("passenger_count", "trip_distance", "fare_amount").filter(col("trip_distance") > 2).explain()

In [ ]:
# An aggregate plan — note the Exchange (shuffle) and partial/final aggregates.
df.groupBy("passenger_count").count().explain()